# Grabbing images from the cameras
#### This takes an image every 0.1 second. 
#### Make sure to change **name** and **seconds**


# Utilities and Declarations


In [6]:

%matplotlib inline
import torch
import json
import os
import cv2
import numpy as np
import time
from datetime import datetime
import glob
from ultralytics import YOLO
import requests
# from shapely.geometry import Polygon
import shutil
from pathlib import Path
import os
import webbrowser


MAIN_DIR = os.getcwd()
IMAGES_SOURCE_DIR = 'dataset/images_with_subfolders'





# record images through video files

In [6]:
# masks are created with draw_mask.py
mask_c042 = [(1, 341), (379, 153), (661, 81), (889, 81), (1050, 190), (946, 279), (1279, 460), (1276, 957), (2, 957)]
mask_c041 = [(1, 236), (132, 191), (462, 98), (587, 87), (829, 78), (986, 90), (1124, 111), (1228, 189), (1091, 279), (1277, 421), (1276, 956), (4, 958)]

import cv2
import os
from datetime import datetime

camera_name = 'c041'
# IMAGES_SOURCE_DIR = "dataset" # Define this if not already set
images_path = f'{IMAGES_SOURCE_DIR}/{camera_name}'

if not os.path.exists(images_path):
    os.makedirs(images_path)

# Initialize Camera

print ('dir: ', os.getcwd())
file_path = f'AICity22_Track1_MTMC_Tracking/test/S06/{camera_name}/vdo.avi'
print ('exists: ', os.path.exists(file_path))

cap = cv2.VideoCapture(file_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_count = 0


next_image_number = 0
while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video or failed to grab frame")
        break


    mask = np.zeros(frame.shape, dtype=np.uint8)
    pts = np.array(mask_c042 if camera_name == 'c042' else mask_c041, dtype=np.int32)
    cv2.fillPoly(mask, [pts], (255, 255, 255))
    frame = cv2.bitwise_and(frame, mask)


    frame_count += 1

    # 2. Only process/save if it's the 1st frame of every second
    if frame_count % fps == 0:
        # frame = cv2.flip(frame, 1)

        # Generate filename
        now = datetime.now()
        file_path = os.path.join(images_path, f"{camera_name}_{next_image_number:04d}.jpg")

        # Save image
        cv2.imwrite(file_path, frame)

        # Visual feedback on the frame
        cv2.putText(frame, f"SAVED: {next_image_number:04d}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        next_image_number += 1
    else:
        # Optional: Add text for frames that are just being skipped/played
        pass
    cv2.imshow('OpenVINO Data Collector', frame)

    # Use waitKey(1) to keep the video playing smoothly
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break



cap.release()
cv2.destroyAllWindows() # Cleans up the window

dir:  C:\ComputerVision\car_multicamera
exists:  True
End of video or failed to grab frame


# Train the model:

In [3]:
from ultralytics import YOLO

In [5]:
import os

yaml_file = os.path.join('dataset', 'data.yaml')

model = YOLO('yolo26m.pt')
MAIN_DIR = os.getcwd()

results = model.train(
    data=yaml_file,
    project= os.path.join(MAIN_DIR, 'runs'),
    epochs=50,
    imgsz=640,
    degrees=15.0,
    shear=2.0,
    perspective=0.001,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    batch=16,
    # patience=10,
)

model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
print(f"The best segmentation model is saved at: {model_path}")

New https://pypi.org/project/ultralytics/8.4.26 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=t